In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.cluster import KMeans
import os
import time

T_HORIZON = 20  # Cycle length (Time steps)
I_MAX = 20      # Max possible inventory
K_TYPES = 20    # Number of discrete customer types (Start with 10)
# Use a reduced set of prices for DP feasibility
PRICES_TO_CHECK = np.linspace(0, 387, 40) 

# File Paths
TRAIN_DATA_PATH = '../data/train_prices_decisions_2025.csv'
MODEL_PATH = 'team-4/xgb_model.pkl'
POLICY_OUTPUT_PATH = 'team-4/dp_policy_table.pkl'
DISCRETIZER_OUTPUT_PATH = 'team-4/customer_discretizer.pkl'

# Load Data and Model
df = pd.read_csv(TRAIN_DATA_PATH)
customer_data = df[['Covariate1', 'Covariate2', 'Covariate3']]
with open(MODEL_PATH, 'rb') as f:
    demand_model = pickle.load(f)

#Discretize the Customer State (c -> k) ---
discretizer = KMeans(n_clusters=K_TYPES, random_state=42, n_init=10)
discretizer.fit(customer_data)

# Save Discretizer
with open(DISCRETIZER_OUTPUT_PATH, 'wb') as f:
    pickle.dump(discretizer, f)

# Get cluster centers for Bellman iteration
cluster_centers = discretizer.cluster_centers_
K_TYPES_ACTUAL = len(cluster_centers)

In [ ]:
# Calculate the Bellman Equation (Backward Recursion)

# Initialize V (Value Function) and PI (Optimal Policy/Price) Tables
V = np.zeros((T_HORIZON + 1, I_MAX + 1, K_TYPES_ACTUAL))
PI = np.zeros((T_HORIZON + 1, I_MAX + 1, K_TYPES_ACTUAL))

start_time = time.time()

for t in range(1, T_HORIZON + 1):  # Time remaining
    for i in range(0, I_MAX + 1):  # Inventory states
        for k in range(0, K_TYPES_ACTUAL):  # Customer Type state
            
            # Base Case: Inventory is Zero
            if i == 0:
                V[t, i, k] = V[t - 1, i, k] 
                continue
            
            best_value = -np.inf
            best_price = 0
            
            # Get the average features for customer type k
            c_k_features = cluster_centers[k]
            
            # Maximize over Price Action Space (P)
            for price in PRICES_TO_CHECK:
                
                # 1. Estimate P(Buy|p, k) using the Demand Model
                X_predict = np.append(c_k_features, price).reshape(1, -1)
                P_buy = demand_model.predict_proba(X_predict)[:, 1][0]
                
                immediate_revenue = price * P_buy
                
                # 2. Bellman Equation: core recursion
                expected_future_value = (
                    P_buy * V[t - 1, i - 1, k]   # If sale: value of next state with i-1 inventory
                    + (1 - P_buy) * V[t - 1, i, k]  # If no sale: value of next state with i inventory
                )
                
                current_value = immediate_revenue + expected_future_value

                # 3. Policy Maximization (Max_p)
                if current_value > best_value:
                    best_value = current_value
                    best_price = price

            V[t, i, k] = best_value
            PI[t, i, k] = best_price # Store the optimal price

# Final Step: Save the Policy (Second required file)
with open(POLICY_OUTPUT_PATH, 'wb') as f:
    pickle.dump(PI, f)
    
print(f"Policy Table shape: {PI.shape}")
print(f"Total Computation Time: {time.time() - start_time:.2f} seconds.")
print(f"DP Agent is ready. Files created: {POLICY_OUTPUT_PATH} and {DISCRETIZER_OUTPUT_PATH}")

Policy Table shape: (21, 21, 20)
Total Computation Time: 33.41 seconds.
DP Agent is ready. Files created: team-4/dp_policy_table.pkl and team-4/customer_discretizer.pkl
